# 04 · Data Pipelines: el flujo ETL en capas

⏱ **12 min** · Módulo 5 · Slide de la Intro: 20 (el flujo ETL y la arquitectura medallion)

**Objetivo:** construir a mano el pipeline completo, de la API a la tabla que usa el negocio.

```
 E · Extracción                T · Transformación             L · Carga
API TVMaze ─► RAW (Volumen) ─► BRONZE ──────────► SILVER ──────────► GOLD
                               crudo, tal         limpio, validado    listo para
                               como llega         y conformado        el negocio
```

| Paso | Qué haces |
|---|---|
| 1 | Extraer: de la API a RAW |
| 2 | Bronze: el dato crudo |
| 3 | Silver: limpio y conformado |
| 4 | Validar antes de publicar |
| 5 | Gold: listo para el negocio |
| 6 | Las tablas nacen gobernadas |
| 7 | Todo junto… y lo que le falta |

In [0]:
%run ./00_Setup

## Paso 1 · Extraer: de la API a RAW

Tres días de programación, un archivo JSON por día, sin tocarlos.

In [0]:
download_days(date(2024, 3, 1), 3)

In [0]:
display(dbutils.fs.ls(RAW_PATH))

## Paso 2 · Bronze: el dato crudo

Leer todos los archivos y guardarlos en una tabla Delta, tal como llegan.

In [0]:
bronze = (spark.read
    .schema(ESQUEMA_EPISODIOS)                 # esquema explícito
    .option("multiLine", "true")               # cada archivo es un arreglo JSON
    .json(RAW_PATH)                            # todos los archivos de la carpeta
    .select("*",
            F.col("_metadata.file_name").alias("source_file"),    # de qué archivo viene cada fila
            F.current_timestamp().alias("ingested_at")))

bronze.write.mode("overwrite").saveAsTable(BRONZE)

In [0]:
%sql
SELECT source_file, count(*) AS episodios
FROM bronze_schedule
GROUP BY source_file
ORDER BY source_file

👀 **Qué observar:** es el código de `load_bronze()` en `00_Setup`. `overwrite` reemplaza la tabla completa: correrlo dos veces
da el mismo resultado. En el laboratorio anterior esta carga era incremental (Auto Loader); aquí lo que importa es el flujo.

## Paso 3 · Silver: limpio y conformado

Lee el código de `build_silver()` antes de ejecutarlo.

In [0]:
print(inspect.getsource(build_silver))

In [0]:
build_silver()

In [0]:
%sql
SELECT * FROM silver_episodes LIMIT 5

👀 **Qué observar:** la transformación aplana el JSON, renombra columnas, convierte `airdate` a fecha, completa la red que falta
y descarta las filas sin id o sin fecha.

## Paso 4 · Validar antes de publicar

In [0]:
validate_silver(min_rows=1)

In [0]:
# ¿Y si una regla no se cumple? La función lanza un error
try:
    validate_silver(min_rows=1_000_000)
except Exception as e:
    print("ERROR →", e)

👀 **Qué observar:** un error detiene lo que venga después. Esa es la idea: si silver está mal, gold no se actualiza.

## Paso 5 · Gold: listo para el negocio

In [0]:
build_gold()

In [0]:
%sql
SELECT * FROM gold_network_daily
ORDER BY episodes DESC
LIMIT 10

## Paso 6 · Las tablas nacen gobernadas

In [0]:
%sql
SELECT table_name, table_type, table_owner, created
FROM information_schema.tables
WHERE table_schema = 'lab_tvmaze_m56' AND table_name IN ('bronze_schedule', 'silver_episodes', 'gold_network_daily')
ORDER BY created

👀 **Qué observar:** las tres tablas ya están en Unity Catalog, con dueño y fecha, en el mismo esquema que gobernamos en el módulo 4.
Abre **Catalog → gold_network_daily → Lineage**: el recorrido `bronze → silver → gold` quedó registrado solo.

## Paso 7 · Todo junto… y lo que le falta

In [0]:
download_days(date(2024, 3, 1), 3)   # E · extraer
load_bronze()                        #     bronze
build_silver()                       # T · transformar
validate_silver()                    #     validar
build_gold()                         # L · cargar

El pipeline funciona, pero lo corres **tú, a mano**:

| Pregunta | Hoy | Lo resuelve |
|---|---|---|
| ¿Quién lo ejecuta cada mañana? | Tú | Un **horario** |
| ¿Y si la API falla un momento? | Lo corres de nuevo | **Reintentos** automáticos |
| ¿Cómo me entero si falló anoche? | Revisando | **Alertas** |
| ¿Y si mañana quiero otra fecha? | Editas el código | **Parámetros** |

Eso es un **Workflow**: el siguiente notebook.

## ✔ Checkpoint

- [ ] Sé qué hace cada etapa: extracción, transformación y carga
- [ ] Sé qué guarda cada capa: bronze (crudo), silver (limpio), gold (negocio)
- [ ] Vi que una validación fallida lanza un error
- [ ] Ejecuté el pipeline completo en cinco líneas

**Siguiente:** `05_Workflows`